In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [12]:
ratings = pd.read_csv(
    "ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="latin-1"
)

ratings.head(10)


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
5,1,1197,3,978302268
6,1,1287,5,978302039
7,1,2804,5,978300719
8,1,594,4,978302268
9,1,919,4,978301368


In [17]:
ratings.shape
ratings['rating'].value_counts().sort_index()


rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

In [5]:
user_item_matrix = ratings.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating',
    fill_value=0
)

user_item_matrix.shape


(6040, 3706)

In [6]:
item_similarity = cosine_similarity(user_item_matrix.T)


In [7]:
movie_ids = user_item_matrix.columns
movie_id_to_index = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}
index_to_movie_id = {idx: movie_id for movie_id, idx in movie_id_to_index.items()}


In [8]:
def recommend_cf(movie_id, ratings, similarity_matrix, movie_id_to_index, top_n=5):

    if movie_id not in movie_id_to_index:
        return "Movie not found in ratings data."

    idx = movie_id_to_index[movie_id]

    sim_scores = list(enumerate(similarity_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    top_indices = [i[0] for i in sim_scores[1:top_n + 1]]
    recommended_movie_ids = [index_to_movie_id[i] for i in top_indices]

    return recommended_movie_ids


In [9]:
movies = pd.read_csv(
    "movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)


In [10]:
def get_movie_titles(movie_ids, movies):
    return movies[movies['movie_id'].isin(movie_ids)][['movie_id', 'title']]


In [11]:
recommended_ids = recommend_cf(
    movie_id=1,
    ratings=ratings,
    similarity_matrix=item_similarity,
    movie_id_to_index=movie_id_to_index,
    top_n=5
)

get_movie_titles(recommended_ids, movies)


,movie_id,title
584,588,Aladdin (1992)
1245,1265,Groundhog Day (1993)
1250,1270,Back to the Future (1985)
2286,2355,"Bug's Life, A (1998)"
3045,3114,Toy Story 2 (1999)


In [ ]:
''' In my implementation, collaborative filtering is item-based rather than user-based.
The input is a movie, and similarity is computed based on how users rated that movie compared to others. 
The notion of “users like me” is implicitly captured through shared rating patterns across users.'''